In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
print('Exists:', os.path.exists(PROJECT_DIR))
print('Contents:', os.listdir(PROJECT_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Exists: False


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ecg-transcovnet'

In [5]:
import os
print('Root of My Drive:')
for item in os.listdir('/content/drive/MyDrive'):
    print(' ', item)

Root of My Drive:
  Colab Notebooks
  ecg-transcovnet


In [6]:
import os, json
import numpy as np
import tensorflow as tf
from collections import Counter
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_class_weight

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')
print('Processed dir exists:', os.path.exists(PROCESSED_DIR))
print('Contents:', os.listdir(PROCESSED_DIR))

def load_split(name):
    d = np.load(os.path.join(PROCESSED_DIR, f'{name}.npz'))
    return d['X'], d['y'], d['R'], d['record_id']

X_train, y_train, R_train, rec_train = load_split('train')
X_val,   y_val,   R_val,   rec_val   = load_split('val')

CLASS_NAMES = ['N', 'S', 'V', 'Q']

def drop_F_and_remap(X, y, R, rec):
    mask = y != 3
    X2, y2, R2, rec2 = X[mask], y[mask].copy(), R[mask], rec[mask]
    y2[y2 == 4] = 3
    return X2, y2, R2, rec2

X_train, y_train, R_train, rec_train = drop_F_and_remap(X_train, y_train, R_train, rec_train)
X_val,   y_val,   R_val,   rec_val   = drop_F_and_remap(X_val,   y_val,   R_val,   rec_val)

X_pool = np.concatenate([X_train, X_val], axis=0)
y_pool = np.concatenate([y_train, y_val], axis=0)
R_pool = np.concatenate([R_train, R_val], axis=0)
groups_pool = np.concatenate([rec_train, rec_val], axis=0)

print('Pool:', X_pool.shape, '| patients:', len(np.unique(groups_pool)), '| classes:', Counter(y_pool))

def per_beat_normalize(X, eps=1e-8):
    m = X.mean(axis=1, keepdims=True)
    s = X.std(axis=1, keepdims=True)
    return (X - m) / (s + eps)

def add_channel(X):
    return X[..., np.newaxis].astype(np.float32)

BATCH_SIZE = 128
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(X, R, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices(({'waveform': X, 'rr_features': R}, y))
    if training:
        ds = ds.shuffle(buffer_size=len(X), seed=42)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)
fold_splits = list(gkf.split(X_pool, y_pool, groups=groups_pool))
for i, (tr_idx, va_idx) in enumerate(fold_splits):
    print(f'Fold {i+1}: val_patients={sorted(set(groups_pool[va_idx]))}')

candidate_configs = [
    {'N': 1.0, 'S': 1.0, 'V': 1.0, 'Q': 1.0},
    {'N': 1.0, 'S': 1.5, 'V': 0.8, 'Q': 0.8},
    {'N': 1.0, 'S': 2.0, 'V': 0.8, 'Q': 0.8},
    {'N': 1.0, 'S': 1.5, 'V': 1.0, 'Q': 1.0},
]

def positional_encoding(length, d_model):
    positions = np.arange(length)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (dims // 2)) / np.float32(d_model))
    angle_rads = positions * angle_rates
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
    return tf.cast(angle_rads[np.newaxis, ...], dtype=tf.float32)

def transformer_block(x, d_model, num_heads=4, ff_dim=128, dropout=0.2, name_prefix='tb'):
    attn_out = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout,
        name=f'{name_prefix}_mha')(x, x)
    x = tf.keras.layers.Add(name=f'{name_prefix}_add1')([x, attn_out])
    x = tf.keras.layers.LayerNormalization(name=f'{name_prefix}_ln1')(x)
    ff = tf.keras.layers.Dense(ff_dim, activation='relu', name=f'{name_prefix}_ff1')(x)
    ff = tf.keras.layers.Dropout(dropout)(ff)
    ff = tf.keras.layers.Dense(d_model, name=f'{name_prefix}_ff2')(ff)
    x = tf.keras.layers.Add(name=f'{name_prefix}_add2')([x, ff])
    x = tf.keras.layers.LayerNormalization(name=f'{name_prefix}_ln2')(x)
    return x

def build_ecg_transcovnet(beat_len, n_rr_features=4, n_classes=4, seed=42,
                           d_model=64, num_heads=4, num_transformer_blocks=2, ff_dim=128):
    tf.keras.utils.set_random_seed(seed)
    reg = tf.keras.regularizers.l2(1e-4)

    wave_in = tf.keras.layers.Input(shape=(beat_len, 1), name='waveform')
    x = tf.keras.layers.Conv1D(32, 7, activation='relu', padding='same', kernel_regularizer=reg)(wave_in)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Conv1D(64, 5, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Conv1D(d_model, 3, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    seq_len = x.shape[1]
    pos_enc = positional_encoding(seq_len, d_model)
    x = tf.keras.layers.Lambda(lambda t: t + pos_enc, name='add_pos_encoding')(x)

    for i in range(num_transformer_blocks):
        x = transformer_block(x, d_model, num_heads=num_heads, ff_dim=ff_dim,
                               dropout=0.2, name_prefix=f'transformer_{i}')

    wave_feat = tf.keras.layers.GlobalAveragePooling1D(name='wave_pool')(x)

    rr_in = tf.keras.layers.Input(shape=(n_rr_features,), name='rr_features')
    r = tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=reg)(rr_in)
    r = tf.keras.layers.BatchNormalization()(r)
    r = tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=reg)(r)

    merged = tf.keras.layers.Concatenate()([wave_feat, r])
    d = tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=reg)(merged)
    d = tf.keras.layers.Dropout(0.4)(d)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(d)

    model = tf.keras.Model(inputs=[wave_in, rr_in], outputs=out, name='ECG_TransCovNet')
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

print('\n✅ Full reset complete.')

Processed dir exists: True
Contents: ['normalization_stats.json', 'train.npz', 'val.npz', 'test.npz', 'train_v2.npz', 'val_v2.npz', 'test_v2.npz', 'preprocessing_config_v2.json']
Pool: (57136, 259) | patients: 25 | classes: Counter({np.int64(0): 46109, np.int64(3): 6230, np.int64(2): 3853, np.int64(1): 944})
Fold 1: val_patients=[np.str_('107'), np.str_('115'), np.str_('116'), np.str_('124'), np.str_('215')]
Fold 2: val_patients=[np.str_('104'), np.str_('122'), np.str_('201'), np.str_('207'), np.str_('209')]
Fold 3: val_patients=[np.str_('102'), np.str_('108'), np.str_('109'), np.str_('119'), np.str_('203')]
Fold 4: val_patients=[np.str_('101'), np.str_('112'), np.str_('205'), np.str_('220'), np.str_('230')]
Fold 5: val_patients=[np.str_('106'), np.str_('114'), np.str_('118'), np.str_('208'), np.str_('223')]

✅ Full reset complete.


In [7]:
import pickle

CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, 'transcovnet_oof_progress.pkl')

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'rb') as f:
            return pickle.load(f)
    return {'completed': {}, 'oof_preds': {cfg_i: np.full(len(y_pool), -1, dtype=np.int64)
                                            for cfg_i in range(len(candidate_configs))}}

def save_checkpoint(state):
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(state, f)

state = load_checkpoint()
print(f'Resuming from checkpoint: {len(state["completed"])} (config, fold) pairs already done.')
if state['completed']:
    print('Already completed:', sorted(state['completed'].keys()))

for cfg_i, cfg in enumerate(candidate_configs):
    print(f'\n===== [TransCovNet] Config {cfg_i+1}/{len(candidate_configs)}: {cfg} =====')

    for fold_i, (tr_idx, va_idx) in enumerate(fold_splits):
        key = (cfg_i, fold_i)
        if key in state['completed']:
            print(f'  Fold {fold_i+1}: SKIPPED (already done, from checkpoint)')
            continue

        X_tr_raw, y_tr, R_tr_raw = X_pool[tr_idx], y_pool[tr_idx], R_pool[tr_idx]
        X_va_raw, y_va, R_va_raw = X_pool[va_idx], y_pool[va_idx], R_pool[va_idx]

        X_tr = per_beat_normalize(X_tr_raw)
        X_va = per_beat_normalize(X_va_raw)

        fold_rr_mean = R_tr_raw.mean(axis=0)
        fold_rr_std  = R_tr_raw.std(axis=0) + 1e-8
        R_tr = (R_tr_raw - fold_rr_mean) / fold_rr_std
        R_va = (R_va_raw - fold_rr_mean) / fold_rr_std

        combined_tr = np.concatenate([X_tr, R_tr], axis=1)
        counts = Counter(y_tr)
        majority = max(counts.values())
        strat = {}
        for cls, cnt in counts.items():
            if cnt < majority:
                strat[cls] = max(min(int(majority * 0.25), majority), cnt)
        min_cnt = min(counts.values())
        k = min(5, max(1, min_cnt - 1))
        sm = SMOTE(random_state=42, k_neighbors=k, sampling_strategy=strat)
        combined_res, y_tr_res = sm.fit_resample(combined_tr, y_tr)

        beat_len_fold = X_tr.shape[1]
        X_tr_res = combined_res[:, :beat_len_fold]
        R_tr_res = combined_res[:, beat_len_fold:]

        train_ds_fold = make_ds(add_channel(X_tr_res), R_tr_res.astype(np.float32),
                                 y_tr_res.astype(np.int64), training=True)

        classes_present_fold = np.unique(y_tr_res)
        base_w = compute_class_weight('balanced', classes=classes_present_fold, y=y_tr_res)
        base_w_dict = dict(zip(classes_present_fold.tolist(), base_w.tolist()))
        cw = {i: base_w_dict[i] * cfg[CLASS_NAMES[i]] for i in classes_present_fold}

        m = build_ecg_transcovnet(beat_len_fold, seed=42)
        m.fit(train_ds_fold, epochs=30,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor='loss', patience=6, restore_best_weights=True)],
              class_weight=cw, verbose=0)

        probs = m.predict({'waveform': add_channel(X_va), 'rr_features': R_va.astype(np.float32)}, verbose=0)
        preds = np.argmax(probs, axis=1)

        state['oof_preds'][cfg_i][va_idx] = preds
        state['completed'][key] = True
        save_checkpoint(state)

        print(f'  Fold {fold_i+1} done and CHECKPOINTED ({len(va_idx)} beats)')

    oof_preds_cfg = state['oof_preds'][cfg_i]
    if (oof_preds_cfg == -1).sum() == 0:
        oof_f1 = f1_score(y_pool, oof_preds_cfg, average='macro')
        print(f'Config {cfg} -> OOF macro-F1: {oof_f1:.4f}')

print('\n\n===== [TransCovNet] FINAL OOF SUMMARY =====')
oof_results_transcovnet = {}
for cfg_i, cfg in enumerate(candidate_configs):
    preds = state['oof_preds'][cfg_i]
    if (preds == -1).sum() == 0:
        f1 = f1_score(y_pool, preds, average='macro')
        oof_results_transcovnet[cfg_i] = (f1, preds)
        print(f'{cfg}: OOF macro-F1={f1:.4f}')
    else:
        print(f'{cfg}: INCOMPLETE ({(preds == -1).sum()} beats still missing)')

if len(oof_results_transcovnet) == len(candidate_configs):
    best_cfg_i_tc = max(oof_results_transcovnet, key=lambda i: oof_results_transcovnet[i][0])
    best_config_tc = candidate_configs[best_cfg_i_tc]
    best_oof_preds_tc = oof_results_transcovnet[best_cfg_i_tc][1]
    print(f'\nBest config (TransCovNet): {best_config_tc}')
    print('\n===== Classification report (OOF, best config, full pool) =====')
    print(classification_report(y_pool, best_oof_preds_tc, target_names=CLASS_NAMES, digits=4, zero_division=0))
else:
    print('\nNot all configs complete yet -- re-run this cell to continue from checkpoint.')

Resuming from checkpoint: 11 (config, fold) pairs already done.
Already completed: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (2, 0)]

===== [TransCovNet] Config 1/4: {'N': 1.0, 'S': 1.0, 'V': 1.0, 'Q': 1.0} =====
  Fold 1: SKIPPED (already done, from checkpoint)
  Fold 2: SKIPPED (already done, from checkpoint)
  Fold 3: SKIPPED (already done, from checkpoint)
  Fold 4: SKIPPED (already done, from checkpoint)
  Fold 5: SKIPPED (already done, from checkpoint)
Config {'N': 1.0, 'S': 1.0, 'V': 1.0, 'Q': 1.0} -> OOF macro-F1: 0.6023

===== [TransCovNet] Config 2/4: {'N': 1.0, 'S': 1.5, 'V': 0.8, 'Q': 0.8} =====
  Fold 1: SKIPPED (already done, from checkpoint)
  Fold 2: SKIPPED (already done, from checkpoint)
  Fold 3: SKIPPED (already done, from checkpoint)
  Fold 4: SKIPPED (already done, from checkpoint)
  Fold 5: SKIPPED (already done, from checkpoint)
Config {'N': 1.0, 'S': 1.5, 'V': 0.8, 'Q': 0.8} -> OOF macro-F1: 0.6320

===== [TransCovNet] Con

In [9]:
def build_two_branch_model(beat_len, n_rr_features=4, n_classes=4, seed=42):
    tf.keras.utils.set_random_seed(seed)
    reg = tf.keras.regularizers.l2(1e-4)

    wave_in = tf.keras.layers.Input(shape=(beat_len, 1), name='waveform')
    x = tf.keras.layers.Conv1D(32, 7, activation='relu', padding='same', kernel_regularizer=reg)(wave_in)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Conv1D(64, 5, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Conv1D(128, 3, activation='relu', padding='same', kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    wave_feat = tf.keras.layers.GlobalAveragePooling1D()(x)

    rr_in = tf.keras.layers.Input(shape=(n_rr_features,), name='rr_features')
    r = tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=reg)(rr_in)
    r = tf.keras.layers.BatchNormalization()(r)
    r = tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=reg)(r)

    merged = tf.keras.layers.Concatenate()([wave_feat, r])
    d = tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=reg)(merged)
    d = tf.keras.layers.Dropout(0.4)(d)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(d)

    model = tf.keras.Model(inputs=[wave_in, rr_in], outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

best_config = {'N': 1.0, 'S': 1.0, 'V': 1.0, 'Q': 1.0}          # Baseline A's winning config
best_config_tc = {'N': 1.0, 'S': 1.5, 'V': 0.8, 'Q': 0.8}       # TransCovNet's winning config
print('Ready.')

Ready.


In [10]:
import pickle

CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
PROB_CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, 'oof_probabilities.pkl')

def load_prob_checkpoint():
    if os.path.exists(PROB_CHECKPOINT_FILE):
        with open(PROB_CHECKPOINT_FILE, 'rb') as f:
            return pickle.load(f)
    return {
        'completed': {},
        'probs_baseline': np.zeros((len(y_pool), 4), dtype=np.float32),
        'probs_transcovnet': np.zeros((len(y_pool), 4), dtype=np.float32),
    }

def save_prob_checkpoint(state):
    with open(PROB_CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(state, f)

pstate = load_prob_checkpoint()
print(f'Resuming: {len(pstate["completed"])} (model, fold) pairs already done.')

MODEL_BUILDERS = {
    'baseline': (build_two_branch_model, best_config, 'probs_baseline'),
    'transcovnet': (build_ecg_transcovnet, best_config_tc, 'probs_transcovnet'),
}

for model_name, (builder_fn, cfg, prob_key) in MODEL_BUILDERS.items():
    print(f'\n===== Model: {model_name} (config: {cfg}) =====')

    for fold_i, (tr_idx, va_idx) in enumerate(fold_splits):
        key = (model_name, fold_i)
        if key in pstate['completed']:
            print(f'  Fold {fold_i+1}: SKIPPED (checkpointed)')
            continue

        X_tr_raw, y_tr, R_tr_raw = X_pool[tr_idx], y_pool[tr_idx], R_pool[tr_idx]
        X_va_raw, y_va, R_va_raw = X_pool[va_idx], y_pool[va_idx], R_pool[va_idx]

        X_tr = per_beat_normalize(X_tr_raw)
        X_va = per_beat_normalize(X_va_raw)

        fold_rr_mean = R_tr_raw.mean(axis=0)
        fold_rr_std  = R_tr_raw.std(axis=0) + 1e-8
        R_tr = (R_tr_raw - fold_rr_mean) / fold_rr_std
        R_va = (R_va_raw - fold_rr_mean) / fold_rr_std

        combined_tr = np.concatenate([X_tr, R_tr], axis=1)
        counts = Counter(y_tr)
        majority = max(counts.values())
        strat = {}
        for cls, cnt in counts.items():
            if cnt < majority:
                strat[cls] = max(min(int(majority * 0.25), majority), cnt)
        min_cnt = min(counts.values())
        k = min(5, max(1, min_cnt - 1))
        sm = SMOTE(random_state=42, k_neighbors=k, sampling_strategy=strat)
        combined_res, y_tr_res = sm.fit_resample(combined_tr, y_tr)

        beat_len_fold = X_tr.shape[1]
        X_tr_res = combined_res[:, :beat_len_fold]
        R_tr_res = combined_res[:, beat_len_fold:]

        train_ds_fold = make_ds(add_channel(X_tr_res), R_tr_res.astype(np.float32),
                                 y_tr_res.astype(np.int64), training=True)

        classes_present_fold = np.unique(y_tr_res)
        base_w = compute_class_weight('balanced', classes=classes_present_fold, y=y_tr_res)
        base_w_dict = dict(zip(classes_present_fold.tolist(), base_w.tolist()))
        cw = {i: base_w_dict[i] * cfg[CLASS_NAMES[i]] for i in classes_present_fold}

        m = builder_fn(beat_len_fold, seed=42)
        m.fit(train_ds_fold, epochs=30,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor='loss', patience=6, restore_best_weights=True)],
              class_weight=cw, verbose=0)

        probs = m.predict({'waveform': add_channel(X_va), 'rr_features': R_va.astype(np.float32)}, verbose=0)
        pstate[prob_key][va_idx] = probs
        pstate['completed'][key] = True
        save_prob_checkpoint(pstate)
        print(f'  Fold {fold_i+1} done and CHECKPOINTED ({len(va_idx)} beats)')

print('\n✅ All probability OOF runs complete (or resumed to completion).')

Resuming: 0 (model, fold) pairs already done.

===== Model: baseline (config: {'N': 1.0, 'S': 1.0, 'V': 1.0, 'Q': 1.0}) =====
  Fold 1 done and CHECKPOINTED (11473 beats)
  Fold 2 done and CHECKPOINTED (11525 beats)
  Fold 3 done and CHECKPOINTED (11441 beats)
  Fold 4 done and CHECKPOINTED (11347 beats)
  Fold 5 done and CHECKPOINTED (11350 beats)

===== Model: transcovnet (config: {'N': 1.0, 'S': 1.5, 'V': 0.8, 'Q': 0.8}) =====
  Fold 1 done and CHECKPOINTED (11473 beats)
  Fold 2 done and CHECKPOINTED (11525 beats)
  Fold 3 done and CHECKPOINTED (11441 beats)
  Fold 4 done and CHECKPOINTED (11347 beats)
  Fold 5 done and CHECKPOINTED (11350 beats)

✅ All probability OOF runs complete (or resumed to completion).


In [11]:
# ============================================================
# Ensemble weight search on DEV (OOF) data only
# ============================================================
probs_baseline = pstate['probs_baseline']
probs_transcovnet = pstate['probs_transcovnet']

print('===== Individual model OOF macro-F1 (sanity check, should match earlier runs) =====')
print('Baseline:   ', f1_score(y_pool, np.argmax(probs_baseline, axis=1), average='macro'))
print('TransCovNet:', f1_score(y_pool, np.argmax(probs_transcovnet, axis=1), average='macro'))

print('\n===== Ensemble weight search (w * baseline + (1-w) * transcovnet) =====')
best_w, best_ens_f1 = None, -1
for w in np.arange(0.0, 1.01, 0.1):
    ensemble_probs = w * probs_baseline + (1 - w) * probs_transcovnet
    ensemble_preds = np.argmax(ensemble_probs, axis=1)
    f1 = f1_score(y_pool, ensemble_preds, average='macro')
    print(f'  w={w:.1f} (baseline weight) -> macro-F1={f1:.4f}')
    if f1 > best_ens_f1:
        best_ens_f1 = f1
        best_w = w

print(f'\nBest ensemble weight: w={best_w:.1f} -> macro-F1={best_ens_f1:.4f}')

best_ensemble_probs = best_w * probs_baseline + (1 - best_w) * probs_transcovnet
best_ensemble_preds = np.argmax(best_ensemble_probs, axis=1)
print('\n===== Classification report (best ensemble, OOF, full pool) =====')
print(classification_report(y_pool, best_ensemble_preds, target_names=CLASS_NAMES, digits=4, zero_division=0))

===== Individual model OOF macro-F1 (sanity check, should match earlier runs) =====
Baseline:    0.6395725643289641
TransCovNet: 0.6134754223025587

===== Ensemble weight search (w * baseline + (1-w) * transcovnet) =====
  w=0.0 (baseline weight) -> macro-F1=0.6135
  w=0.1 (baseline weight) -> macro-F1=0.6163
  w=0.2 (baseline weight) -> macro-F1=0.6207
  w=0.3 (baseline weight) -> macro-F1=0.6263
  w=0.4 (baseline weight) -> macro-F1=0.6341
  w=0.5 (baseline weight) -> macro-F1=0.6451
  w=0.6 (baseline weight) -> macro-F1=0.6410
  w=0.7 (baseline weight) -> macro-F1=0.6418
  w=0.8 (baseline weight) -> macro-F1=0.6411
  w=0.9 (baseline weight) -> macro-F1=0.6401
  w=1.0 (baseline weight) -> macro-F1=0.6396

Best ensemble weight: w=0.5 -> macro-F1=0.6451

===== Classification report (best ensemble, OOF, full pool) =====
              precision    recall  f1-score   support

           N     0.9726    0.8460    0.9049     46109
           S     0.1489    0.3422    0.2075       944
      

In [12]:
# ============================================================
# Train final TransCovNet on FULL pool (never done yet -- only fold models existed)
# ============================================================
X_pool_n = per_beat_normalize(X_pool)
pool_rr_mean = R_pool.mean(axis=0)
pool_rr_std  = R_pool.std(axis=0) + 1e-8
R_pool_n = (R_pool - pool_rr_mean) / pool_rr_std

combined_pool = np.concatenate([X_pool_n, R_pool_n], axis=1)
counts = Counter(y_pool)
majority = max(counts.values())
strat = {}
for cls, cnt in counts.items():
    if cnt < majority:
        strat[cls] = max(min(int(majority * 0.25), majority), cnt)
min_cnt = min(counts.values())
k = min(5, max(1, min_cnt - 1))
sm = SMOTE(random_state=42, k_neighbors=k, sampling_strategy=strat)
combined_res, y_pool_res = sm.fit_resample(combined_pool, y_pool)

beat_len_final = X_pool_n.shape[1]
X_pool_res = combined_res[:, :beat_len_final]
R_pool_res = combined_res[:, beat_len_final:]

train_ds_final = make_ds(add_channel(X_pool_res), R_pool_res.astype(np.float32),
                          y_pool_res.astype(np.int64), training=True)

classes_present = np.unique(y_pool_res)
base_w = compute_class_weight('balanced', classes=classes_present, y=y_pool_res)
base_w_dict = dict(zip(classes_present.tolist(), base_w.tolist()))
final_class_weight_tc = {i: base_w_dict[i] * best_config_tc[CLASS_NAMES[i]] for i in classes_present}

final_model_tc = build_ecg_transcovnet(beat_len_final, seed=42)
final_model_tc.fit(train_ds_final, epochs=30,
                    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='loss', patience=6, restore_best_weights=True)],
                    class_weight=final_class_weight_tc, verbose=1)

final_model_tc.save(os.path.join(PROJECT_DIR, 'models', 'ecg_transcovnet_final_locked.keras'))
print('TransCovNet final model saved.')

Epoch 1/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 38s 32ms/step - accuracy: 0.8890 - loss: 0.3124
Epoch 2/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.9707 - loss: 0.0965
Epoch 3/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9805 - loss: 0.0731
Epoch 4/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.9848 - loss: 0.0612
Epoch 5/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.9869 - loss: 0.0549
Epoch 6/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9879 - loss: 0.0510
Epoch 7/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.9896 - loss: 0.0455
Epoch 8/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.9902 - loss: 0.0434
Epoch 9/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9902 - loss: 0.0411
Epoch 10/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.9921 - loss: 0.0358
Epoch 11/30
631/631 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.9924 - loss: 0.0350
Epoch 12/30
631/631 ━━━━━━━━━━━━━━━━━━

In [15]:
# ============================================================
# 🔒 FINAL LOCKED TEST EVALUATION — Baseline A vs TransCovNet vs Ensemble
# This is the ONE, official, paper-ready comparison. No re-tuning after this.
# ============================================================

# Load test set fresh
X_test, y_test, R_test, rec_test = load_split('test')
X_test, y_test, R_test, rec_test = drop_F_and_remap(X_test, y_test, R_test, rec_test)
X_test_n = per_beat_normalize(X_test)
R_test_n = (R_test - pool_rr_mean) / pool_rr_std   # same pool_rr_mean/std used to train both final models

# Load both final models (safe_mode=False needed because ECG-TransCovNet uses a
# Lambda layer for positional encoding -- this is our own trusted file, so it's safe)
final_model_baseline = tf.keras.models.load_model(
    os.path.join(PROJECT_DIR, 'models', 'baseline_a_final_locked.keras'),
    safe_mode=False
)
final_model_tc = tf.keras.models.load_model(
    os.path.join(PROJECT_DIR, 'models', 'ecg_transcovnet_final_locked.keras'),
    safe_mode=False
)

test_inputs = {'waveform': add_channel(X_test_n), 'rr_features': R_test_n.astype(np.float32)}
probs_test_baseline = final_model_baseline.predict(test_inputs, verbose=0)
probs_test_tc = final_model_tc.predict(test_inputs, verbose=0)

# Ensemble weight LOCKED from dev/OOF search (best_w = 0.5) -- not re-searched on test
probs_test_ensemble = best_w * probs_test_baseline + (1 - best_w) * probs_test_tc

results_summary = {}

for name, probs in [('Baseline A', probs_test_baseline),
                     ('ECG-TransCovNet', probs_test_tc),
                     (f'Ensemble (w={best_w:.1f})', probs_test_ensemble)]:
    preds = np.argmax(probs, axis=1)
    macro_f1 = f1_score(y_test, preds, average='macro')
    weighted_f1 = f1_score(y_test, preds, average='weighted')
    acc = (preds == y_test).mean()
    results_summary[name] = {'accuracy': acc, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1}

    print(f'\n{"="*70}\n🔒 {name} — LOCKED TEST RESULTS\n{"="*70}')
    print(classification_report(y_test, preds, target_names=CLASS_NAMES, digits=4, zero_division=0))
    print(f'Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f} | Weighted-F1: {weighted_f1:.4f}')

    cm = confusion_matrix(y_test, preds)
    print('\nConfusion matrix:')
    print('     ', '  '.join(f'{c:>5s}' for c in CLASS_NAMES))
    for i, row in enumerate(cm):
        print(f'{CLASS_NAMES[i]:>5s}', '  '.join(f'{v:5d}' for v in row))

    y_test_onehot = tf.keras.utils.to_categorical(y_test, num_classes=4)
    print('\nPer-class AUROC / AUPRC:')
    for i, cname in enumerate(CLASS_NAMES):
        auroc = roc_auc_score(y_test_onehot[:, i], probs[:, i])
        auprc = average_precision_score(y_test_onehot[:, i], probs[:, i])
        print(f'  {cname}: AUROC={auroc:.4f}  AUPRC={auprc:.4f}')

print(f'\n\n{"="*70}\n📊 FINAL SUMMARY TABLE\n{"="*70}')
print(f'{"Model":<25} {"Accuracy":<12} {"Macro-F1":<12} {"Weighted-F1":<12}')
for name, r in results_summary.items():
    print(f'{name:<25} {r["accuracy"]:<12.4f} {r["macro_f1"]:<12.4f} {r["weighted_f1"]:<12.4f}')

# Save everything
with open(os.path.join(PROJECT_DIR, 'outputs', 'final_locked_results.json'), 'w') as f:
    json.dump({k: {kk: float(vv) for kk, vv in v.items()} for k, v in results_summary.items()}, f, indent=2)

print('\n✅ ALL RESULTS LOCKED AND SAVED. This is the official paper comparison table.')
print('🚫 Do not re-tune, re-select, or re-evaluate on this test set after this point.')

NotImplementedError: Exception encountered when calling Lambda.call().

[1mWe could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.[0m

Arguments received by Lambda.call():
  • args=('<KerasTensor shape=(None, 64, 64), dtype=float32, sparse=False, ragged=False, name=keras_tensor_797>',)
  • kwargs={'mask': 'None'}

In [16]:
# Load test set fresh
X_test, y_test, R_test, rec_test = load_split('test')
X_test, y_test, R_test, rec_test = drop_F_and_remap(X_test, y_test, R_test, rec_test)
X_test_n = per_beat_normalize(X_test)
R_test_n = (R_test - pool_rr_mean) / pool_rr_std

# Baseline A: load from disk (works fine, no Lambda layer issue)
final_model_baseline = tf.keras.models.load_model(
    os.path.join(PROJECT_DIR, 'models', 'baseline_a_final_locked.keras'),
    safe_mode=False
)

# TransCovNet: DO NOT reload from disk -- reuse the `final_model_tc` object
# already trained and sitting in memory from the training cell above.
# (If you restarted the runtime since then, see the note below the code.)

test_inputs = {'waveform': add_channel(X_test_n), 'rr_features': R_test_n.astype(np.float32)}
probs_test_baseline = final_model_baseline.predict(test_inputs, verbose=0)
probs_test_tc = final_model_tc.predict(test_inputs, verbose=0)

probs_test_ensemble = best_w * probs_test_baseline + (1 - best_w) * probs_test_tc

results_summary = {}

for name, probs in [('Baseline A', probs_test_baseline),
                     ('ECG-TransCovNet', probs_test_tc),
                     (f'Ensemble (w={best_w:.1f})', probs_test_ensemble)]:
    preds = np.argmax(probs, axis=1)
    macro_f1 = f1_score(y_test, preds, average='macro')
    weighted_f1 = f1_score(y_test, preds, average='weighted')
    acc = (preds == y_test).mean()
    results_summary[name] = {'accuracy': acc, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1}

    print(f'\n{"="*70}\n🔒 {name} — LOCKED TEST RESULTS\n{"="*70}')
    print(classification_report(y_test, preds, target_names=CLASS_NAMES, digits=4, zero_division=0))
    print(f'Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f} | Weighted-F1: {weighted_f1:.4f}')

    cm = confusion_matrix(y_test, preds)
    print('\nConfusion matrix:')
    print('     ', '  '.join(f'{c:>5s}' for c in CLASS_NAMES))
    for i, row in enumerate(cm):
        print(f'{CLASS_NAMES[i]:>5s}', '  '.join(f'{v:5d}' for v in row))

    y_test_onehot = tf.keras.utils.to_categorical(y_test, num_classes=4)
    print('\nPer-class AUROC / AUPRC:')
    for i, cname in enumerate(CLASS_NAMES):
        auroc = roc_auc_score(y_test_onehot[:, i], probs[:, i])
        auprc = average_precision_score(y_test_onehot[:, i], probs[:, i])
        print(f'  {cname}: AUROC={auroc:.4f}  AUPRC={auprc:.4f}')

print(f'\n\n{"="*70}\n📊 FINAL SUMMARY TABLE\n{"="*70}')
print(f'{"Model":<25} {"Accuracy":<12} {"Macro-F1":<12} {"Weighted-F1":<12}')
for name, r in results_summary.items():
    print(f'{name:<25} {r["accuracy"]:<12.4f} {r["macro_f1"]:<12.4f} {r["weighted_f1"]:<12.4f}')

with open(os.path.join(PROJECT_DIR, 'outputs', 'final_locked_results.json'), 'w') as f:
    json.dump({k: {kk: float(vv) for kk, vv in v.items()} for k, v in results_summary.items()}, f, indent=2)

print('\n✅ ALL RESULTS LOCKED AND SAVED.')


🔒 Baseline A — LOCKED TEST RESULTS
              precision    recall  f1-score   support

           N     0.9639    0.8689    0.9139     44483
           S     0.1738    0.1611    0.1672      1837
           V     0.8516    0.9539    0.8999      3382
           Q     0.2973    0.9729    0.4555      1809

    accuracy                         0.8529     51511
   macro avg     0.5717    0.7392    0.6091     51511
weighted avg     0.9049    0.8529    0.8703     51511

Accuracy: 0.8529 | Macro-F1: 0.6091 | Weighted-F1: 0.8703

Confusion matrix:
          N      S      V      Q
    N 38652   1339    346   4146
    S  1334    296    194     13
    V    88     68   3226      0
    Q    27      0     22   1760

Per-class AUROC / AUPRC:
  N: AUROC=0.9053  AUPRC=0.9808
  S: AUROC=0.7437  AUPRC=0.1413
  V: AUROC=0.9951  AUPRC=0.9720
  Q: AUROC=0.9848  AUPRC=0.6980

🔒 ECG-TransCovNet — LOCKED TEST RESULTS
              precision    recall  f1-score   support

           N     0.9689    0.9454    

In [17]:
import json

results_file = os.path.join(PROJECT_DIR, 'outputs', 'final_locked_results.json')
with open(results_file) as f:
    results = json.load(f)
print(json.dumps(results, indent=2))

{
  "Baseline A": {
    "accuracy": 0.8529052047135564,
    "macro_f1": 0.6091276059251728,
    "weighted_f1": 0.8702788214231592
  },
  "ECG-TransCovNet": {
    "accuracy": 0.9138630583758809,
    "macro_f1": 0.6461978306829849,
    "weighted_f1": 0.9097306110573686
  },
  "Ensemble (w=0.5)": {
    "accuracy": 0.9084467395313622,
    "macro_f1": 0.6575453460788252,
    "weighted_f1": 0.9081549070856973
  }
}


In [18]:
# ============================================================
# COMPLETE VERIFICATION — Naya Colab (Aaj Ka Kaam)
# ============================================================
import os, json
from google.colab import drive

# Mount Drive (agar mounted nahi hai)
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'

print("=" * 70)
print("COMPLETE VERIFICATION — NAYA DRIVE")
print("=" * 70)

# ============================================================
# 1. Top-level folders check
# ============================================================
print("\n📁 TOP-LEVEL FOLDERS:")
if os.path.exists(PROJECT_DIR):
    for item in sorted(os.listdir(PROJECT_DIR)):
        item_path = os.path.join(PROJECT_DIR, item)
        if os.path.isdir(item_path):
            count = len(os.listdir(item_path))
            print(f"   ✅ {item}/ ({count} items)")
        else:
            size = os.path.getsize(item_path) / 1024
            print(f"   📄 {item}: {size:.1f} KB")
else:
    print(f"   ❌ Project not found: {PROJECT_DIR}")

# ============================================================
# 2. Final Models
# ============================================================
print("\n" + "=" * 70)
print("📁 FINAL MODELS")
print("=" * 70)

models_path = os.path.join(PROJECT_DIR, 'models')
if os.path.exists(models_path):
    for f in sorted(os.listdir(models_path)):
        fpath = os.path.join(models_path, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / 1024 / 1024
            print(f"   ✅ {f}: {size:.2f} MB")
        else:
            print(f"   📁 {f}/")
else:
    print(f"   ❌ models/ not found")

# ============================================================
# 3. Final Results
# ============================================================
print("\n" + "=" * 70)
print("📁 FINAL RESULTS")
print("=" * 70)

outputs_path = os.path.join(PROJECT_DIR, 'outputs')
if os.path.exists(outputs_path):
    for f in sorted(os.listdir(outputs_path)):
        fpath = os.path.join(outputs_path, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / 1024
            print(f"   ✅ {f}: {size:.1f} KB")
    # Display results content
    results_file = os.path.join(outputs_path, 'final_locked_results.json')
    if os.path.exists(results_file):
        with open(results_file) as f:
            results = json.load(f)
        print(f"\n   📊 FINAL RESULTS CONTENT:")
        print(f"   {json.dumps(results, indent=4)}")
else:
    print(f"   ❌ outputs/ not found")

# ============================================================
# 4. Data + Splits
# ============================================================
print("\n" + "=" * 70)
print("📁 DATA + SPLITS")
print("=" * 70)

# Data
data_path = os.path.join(PROJECT_DIR, 'data')
if os.path.exists(data_path):
    print(f"\n   📁 data/:")
    for sub in sorted(os.listdir(data_path)):
        sub_path = os.path.join(data_path, sub)
        if os.path.isdir(sub_path):
            count = len(os.listdir(sub_path))
            print(f"      📁 {sub}/ ({count} items)")

# Splits
splits_path = os.path.join(PROJECT_DIR, 'splits')
if os.path.exists(splits_path):
    print(f"\n   📁 splits/:")
    for f in sorted(os.listdir(splits_path)):
        size = os.path.getsize(os.path.join(splits_path, f)) / 1024
        print(f"      📄 {f}: {size:.1f} KB")

# Audit
audit_path = os.path.join(PROJECT_DIR, 'audit')
if os.path.exists(audit_path):
    print(f"\n   📁 audit/:")
    for f in sorted(os.listdir(audit_path)):
        size = os.path.getsize(os.path.join(audit_path, f)) / 1024
        print(f"      📄 {f}: {size:.1f} KB")

# Checkpoints
checkpoints_path = os.path.join(PROJECT_DIR, 'checkpoints')
if os.path.exists(checkpoints_path):
    print(f"\n   📁 checkpoints/:")
    for f in sorted(os.listdir(checkpoints_path)):
        size = os.path.getsize(os.path.join(checkpoints_path, f)) / 1024
        print(f"      📄 {f}: {size:.1f} KB")

# Archive
archive_path = os.path.join(PROJECT_DIR, 'archive_old_experiments')
if os.path.exists(archive_path):
    print(f"\n   📁 archive_old_experiments/:")
    for f in sorted(os.listdir(archive_path)):
        print(f"      📄 {f}")

print("\n" + "=" * 70)
print("✅ VERIFICATION COMPLETE")
print("=" * 70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
COMPLETE VERIFICATION — NAYA DRIVE

📁 TOP-LEVEL FOLDERS:
   ✅ archive_old_experiments/ (2 items)
   ✅ audit/ (4 items)
   ✅ backups/ (4 items)
   ✅ checkpoints/ (3 items)
   ✅ data/ (3 items)
   ✅ models/ (2 items)
   ✅ outputs/ (1 items)
   ✅ splits/ (4 items)

📁 FINAL MODELS
   ✅ baseline_a_final_locked.keras: 0.62 MB
   ✅ ecg_transcovnet_final_locked.keras: 1.38 MB

📁 FINAL RESULTS
   ✅ final_locked_results.json: 0.4 KB

   📊 FINAL RESULTS CONTENT:
   {
    "Baseline A": {
        "accuracy": 0.8529052047135564,
        "macro_f1": 0.6091276059251728,
        "weighted_f1": 0.8702788214231592
    },
    "ECG-TransCovNet": {
        "accuracy": 0.9138630583758809,
        "macro_f1": 0.6461978306829849,
        "weighted_f1": 0.9097306110573686
    },
    "Ensemble (w=0.5)": {
        "accuracy": 0.9084467395313622,
        "macro_f1": 0.6575453460788252,
 